In [1]:
import sys
import os
import torch
sys.path.append('/workspaces/BO_EXPERIMENTS/src')

# 設定設備與型別
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
torch.set_default_dtype(dtype)

import matplotlib.pyplot as plt
import math
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from tqdm import tqdm

from torch.utils.data import DataLoader, TensorDataset, random_split
from botorch.models import MultiTaskGP, KroneckerMultiTaskGP,  SingleTaskGP, ModelListGP
from botorch.models.transforms import Standardize, Normalize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood, SumMarginalLogLikelihood
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.hypervolume import Hypervolume
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.multi_objective.box_decompositions.non_dominated import NondominatedPartitioning
from botorch.utils.multi_objective.box_decompositions.dominated import DominatedPartitioning
from botorch.acquisition.multi_objective import qExpectedHypervolumeImprovement
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.utils.transforms import unnormalize, standardize
from pprint import pprint
from typing import Optional
from contextlib import redirect_stdout

from utils.beta_oracle import load_beta_oracle, oracle_eval, noise_variance
from utils.decomposition import RectangleSphericalVariablesChange




/home/appuser/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def format_multitask_data(X, Y):
    """
    將 X (N, D) 與 Y (N, M) 轉換為 MultiTaskGP 所需格式
    X_out: (N*M, D+1)
    Y_out: (N*M, 1)
    """
    N, D = X.shape
    M = Y.shape[1] # 目標維度，例如 100

    # 1. 處理 X: 將原始 X 重複 M 次 (區塊重複)
    # [x1, x2...xn] -> [x1, x2...xn, x1, x2...xn, ...]
    X_repeated = X.repeat(M, 1)

    # 2. 處理 Task Index: 建立 [0,0...0, 1,1...1, ..., M-1...M-1] 的索引
    # 這裡使用 repeat_interleave 確保每個索引對應一整組 N 個數據
    task_indices = torch.arange(M, device=X.device, dtype=X.dtype).repeat_interleave(N).unsqueeze(-1)

    # 3. 拼接 X 與 Task Index -> (N*M, D+1)
    full_X = torch.cat([X_repeated, task_indices], dim=-1)

    # 4. 處理 Y: 將 (N, M) 轉置後拉平為 (N*M, 1)
    # 注意：必須先轉置 (.t()) 確保順序與 task_indices 對齊
    full_Y = Y.t().reshape(-1, 1)

    return full_X, full_Y

def initialize_independent_gp(train_x, train_y):
    """
    train_x: (N, D)
    train_y: (N, M) -> M 是目標數量
    """
    num_objectives = train_y.shape[-1]
    models = []
    
    for i in range(num_objectives):
        # 提取第 i 個目標的數據
        target_y = train_y[:, i : i + 1] 
        
        # 為每個目標建立獨立的 SingleTaskGP
        # 這裡會自動為每個目標選擇最佳的 Kernel 參數
        model = SingleTaskGP(train_x, target_y, outcome_transform=Standardize(m=1),)
        models.append(model)
    
    # 使用 ModelListGP 將所有模型組合起來
    # 這樣採集函數（如 LogNEHVI）才能同時看到所有目標的預測
    model_list = ModelListGP(*models)
    
    # 對應的 MLL 也要使用 SumMarginalLogLikelihood
    mll = SumMarginalLogLikelihood(model_list.likelihood, model_list)

    # fit model
    with open(os.devnull, 'w') as f:
        with redirect_stdout(f):
            fit_gpytorch_mll(mll, max_retries=50)
    
    return mll, model_list

# 計算超體積
def get_current_hv(train_Y, ref_point):
    # 1. 取得 Pareto Front
    pareto_y = train_Y[is_non_dominated(train_Y)]
    
    # 2. 初始化計算器 (注意大小寫，通常是 Hypervolume)
    hv_obj = Hypervolume(ref_point=ref_point)
    
    # 3. 計算並回傳
    return hv_obj.compute(pareto_y)

def oracle_function(X, oracle):
    intercept = oracle["intercept"]
    beta_lin = oracle["beta_lin"]
    pairs = oracle["pairs"]
    beta_inter = oracle["beta_inter"]

    lin = X @ beta_lin.T
    cross = X[:, pairs[:, 0]] * X[:, pairs[:, 1]]
    inter = cross @ beta_inter.T
    return intercept.unsqueeze(0) + lin + inter

def load_dataset(data_info_path, target_cols=['SPGR', 'TE']):
    # 讀取資料 'target_cols': ['SPGR', 'TE']
    if isinstance(data_info_path, str):
        if os.path.splitext(data_info_path)[-1] == '.pt':
            data_info = torch.load(data_info_path)
        elif os.path.splitext(data_info_path)[-1] == '.csv':
            ori_data = pd.read_csv(data_info_path)
            feature_cols = [c for c in list(ori_data.columns) if c not in target_cols]
            x = torch.tensor(ori_data.loc[:, feature_cols].to_numpy(), device=device)
            y = torch.tensor(ori_data.loc[:, target_cols].to_numpy(), device=device)
            data_info = {
                'X': x/100,
                'Y': y,
                'feature_cols': feature_cols,
                'target_cols': target_cols
            }
    elif isinstance(data_info_path, list):
        data_info = []
        for i, p in enumerate(data_info_path):
            train_ori_data = pd.read_csv(p)
            test_ori_data = pd.concat([pd.read_csv(test_p) for test_p in data_info_path if test_p != p])
            feature_cols = [c for c in list(train_ori_data.columns) if c not in target_cols]
            train_x = torch.tensor(train_ori_data.loc[:, feature_cols].to_numpy(), device=device)
            train_y = torch.tensor(train_ori_data.loc[:, target_cols].to_numpy(), device=device)
            test_x = torch.tensor(test_ori_data.loc[:, feature_cols].to_numpy(), device=device)
            test_y = torch.tensor(test_ori_data.loc[:, target_cols].to_numpy(), device=device)

            data_info.append(
                {
                    'train': {
                        'X': train_x / 100,
                        'Y': train_y,
                        'feature_cols': feature_cols,
                        'target_cols': target_cols
                    },
                    'test': {
                        'X': test_x / 100,
                        'Y': test_y,
                        'feature_cols': feature_cols,
                        'target_cols': target_cols
                    }
                }
            )

    return data_info


In [3]:
# 設定結果儲存位置
result_save_dir = '/workspaces/BO_EXPERIMENTS/src/results/20260309/decomposition_bo'

# 設定 dataset 位置
# data_info_path = '/workspaces/BO_EXPERIMENTS/src/datasets/sparse_dataset.pt'
# data_info_path = '/workspaces/BO_EXPERIMENTS/src/datasets/20260203_data_47d_spgr_te.csv'
data_info_path = [
    '/workspaces/BO_EXPERIMENTS/src/datasets/synthetic_data_sparse_constraint_seed_39.csv',
    '/workspaces/BO_EXPERIMENTS/src/datasets/synthetic_data_sparse_constraint_seed_40.csv',
    '/workspaces/BO_EXPERIMENTS/src/datasets/synthetic_data_sparse_constraint_seed_41.csv'
]

# 設定 beta 係數的位置
beta_path = '/workspaces/BO_EXPERIMENTS/src/datasets/beta1.csv'

# 設定 iteration 次數
n_iter = 50

# 設定 seed ls
random_state_ls = [39, 40, 41, 42]#[1000, 523, 4456, 21]

# 設定 training dataset 的比例
train_ratio = 0.2

# 設定 reference point
ref_point = torch.tensor([1.3654, 2.8482], dtype=dtype, device=device)

# 設定 target_cols
target_cols = ['SPGR', 'TE']

# 設定是否要進行極坐標轉換
decomposition = True

# 建立儲存位置
os.makedirs(result_save_dir, exist_ok=True)

# 讀取資料 'target_cols': ['SPGR', 'TE']
# if os.path.splitext(data_info_path)[-1] == '.pt':
#     data_info = torch.load(data_info_path)
# elif os.path.splitext(data_info_path)[-1] == '.csv':
#     ori_data = pd.read_csv(data_info_path)
#     feature_cols = [c for c in list(ori_data.columns) if c not in target_cols]
#     x = torch.tensor(ori_data.loc[:, feature_cols].to_numpy(), device=device)
#     y = torch.tensor(ori_data.loc[:, target_cols].to_numpy(), device=device)
#     data_info = {
#         'X': x,
#         'Y': y,
#         'feature_cols': feature_cols,
#         'target_cols': target_cols
#     }
data_info = load_dataset(data_info_path, target_cols=target_cols)

In [4]:
# pt_path = '/workspaces/BO_EXPERIMENTS/src/datasets/sparse_dataset.pt'
# csv_path = '/workspaces/BO_EXPERIMENTS/src/datasets/20260203_data_47d_spgr_te.csv'

# data_info_pt = load_dataset(pt_path, target_cols=target_cols)
# data_info_csv = load_dataset(csv_path, target_cols=target_cols)

# pd.DataFrame(data_info_csv['X'].cpu().numpy())
# pd.DataFrame(data_info_pt['X'].cpu().numpy())

In [5]:
# 讀取 oracle function
oracle = load_beta_oracle(beta_path, device=device, dtype=dtype)

# 準備資料 (如果 data_info 是一個 dict 才要做這一個流程)
if isinstance(data_info, dict):
    total_x = data_info['X']
    total_std_x = total_x
    total_y = data_info['Y']

    train_size = 200 #int(total_x.shape[0]*train_ratio)
    test_size = total_x.shape[0] - train_size

    dataset = TensorDataset(total_std_x, total_y)

In [ ]:
# rd_idx = 0
random_state_ls = random_state_ls if isinstance(data_info, dict) else list(range(len(data_info) ))
for rd_idx in range(len(random_state_ls)):
    if isinstance(data_info, dict):
        # 設定 generator seed
        generator = torch.Generator().manual_seed(random_state_ls[rd_idx])

        train_dataset, test_dataset = random_split(
            dataset, 
            [train_size, test_size], 
            generator=generator
        )

        train_x = train_dataset.dataset.tensors[0][train_dataset.indices]
        train_y = train_dataset.dataset.tensors[1][train_dataset.indices]
        test_x = test_dataset.dataset.tensors[0][test_dataset.indices]
        test_y = test_dataset.dataset.tensors[1][test_dataset.indices]
    elif isinstance(data_info, list):
        train_x = data_info[rd_idx]['train']['X']
        train_y = data_info[rd_idx]['train']['Y']
        test_x = data_info[rd_idx]['test']['X']
        test_y = data_info[rd_idx]['test']['Y']
    
    t = tqdm(range(n_iter), ncols=80)
    current_hvs = []
    for i in t:
        if decomposition:
            # 對 X 進行極坐標轉換
            theta_train_x = RectangleSphericalVariablesChange(do_x_sqrt=True, return_r=False).to_theta(train_x)
            D = theta_train_x.shape[1]

            theta_train_std_x = theta_train_x / (torch.pi /2)
            # 每個Y獨立定義 GP 並且 fit 這個 model
            mll, model = initialize_independent_gp(theta_train_std_x, train_y)

            # 計算MSE
            theta_test_x = RectangleSphericalVariablesChange(do_x_sqrt=True, return_r=False).to_theta(test_x)
            theta_test_std_x = theta_test_x / (torch.pi /2)
            with torch.no_grad():
                # 取得後驗分佈
                posterior = model.posterior(theta_test_std_x)
                mean = posterior.mean
                mse = torch.mean((test_y - mean)**2)
        else:
            D = train_x.shape[1]
            # 每個Y獨立定義 GP 並且 fit 這個 model
            mll, model = initialize_independent_gp(train_x, train_y)

            # 計算MSE
            with torch.no_grad():
                # 取得後驗分佈
                posterior = model.posterior(test_x)
                mean = posterior.mean
                mse = torch.mean((test_y - mean)**2)

        sampler = SobolQMCNormalSampler(sample_shape=torch.Size([128]))

        # 使用 qLogNoisyExpectedHypervolumeImprovement
        # 優點：不需要手動 partitioning，避開了那個維度報錯的 bug    
        acq_fun = qLogNoisyExpectedHypervolumeImprovement(
            model=model,
            ref_point=ref_point,
            X_baseline=theta_train_std_x if decomposition else train_x, # 使用已有的點作為基準
            prune_baseline=True, # 自動篩選 Pareto 點，避免維度爆炸
            sampler=sampler
        )

        # set bound
        bounds=torch.stack([
            torch.zeros(D, device=device),
            torch.ones(D, device=device),# * torch.pi / 2,
        ])

        if decomposition:
            equality_constraints = None
        else:
            # Set constraints
            equality_constraints = [
                (
                    torch.arange(D, device=device), # indices: X 的哪些維度要參與計算
                    torch.ones(D, dtype=dtype, device=device), # coefficients: 這些維度的係數
                    torch.tensor([1.0], device=device, dtype=dtype) # rhs: 等號右邊的值 (Sum = 1.0)
                )
            ]

        # 開始執行最佳化
        new_x, _ = optimize_acqf(
            acq_function = acq_fun, 
            bounds = bounds, 
            q= 1, 
            num_restarts = 10, 
            raw_samples = 128, 
            equality_constraints = equality_constraints,
            options={
                "batch_limit": 5, 
                "maxiter": 200
            }
        )

        if decomposition:
            new_x = RectangleSphericalVariablesChange(do_x_sqrt=True, return_r=False).to_x(new_x * (torch.pi/2))

        # 推薦點放到 oracle 取得新的觀察值
        new_y = oracle_function(new_x, oracle)
        # oracle_eval(new_x, oracle)

        # 與舊的資料合併
        train_x = torch.cat([train_x, new_x], dim=0)
        train_y = torch.cat([train_y, new_y], dim=0)

        # Compute HV
        # ============================
        # 1. 篩選出 Pareto 前沿
        pareto_mask = is_non_dominated(train_y)
        pareto_front = train_y[pareto_mask]

        # 初始化並計算
        hv_calc = Hypervolume(ref_point=ref_point)
        hv = hv_calc.compute(pareto_front)
        current_hvs.append(hv)

        message = {'MSE': mse.cpu().item(), 'HV': hv, 'NumData': train_y.shape[0]}

        t.set_postfix(**message)

    # 儲存 HV ls
    save_data = {
        'hv_ls': current_hvs,
        'datasplit_seed': random_state_ls[rd_idx]
    }
    if decomposition:
        file_name = 'DecompositionBO_HVs_with_dataset_split_seed_{}.pkl'.format(random_state_ls[rd_idx])
    else:
        file_name = 'BO_HVs_with_dataset_split_seed_{}.pkl'.format(random_state_ls[rd_idx])
    result_save_path = os.path.join(result_save_dir, file_name)
    joblib.dump(save_data, result_save_path)

  2%|     | 1/50 [00:08<07:13,  8.85s/it, HV=0.00424, MSE=0.000295, NumData=101]


KeyboardInterrupt: 

In [ ]:
acq_fun